# agentsafe + Ollama + Local OpenTelemetry demo

This notebook shows a decorator-first integration, so you can keep your existing OpenAI/Ollama function and add safety with one line.

1. Load `NVIDIA_API_KEY` from `.env`
2. Keep your existing OpenAI SDK call to local Ollama
3. Add `@safe(...)` on top of that function
4. Enable OpenTelemetry export to localhost
5. Verify blocking behavior on unsafe prompts
6. Inspect observability metrics and traces in the notebook
7. View Jaeger UI embedded directly inside notebook output

## Prerequisites

- Ollama running on `http://localhost:11434`
- A pulled model such as `llama3.2:3b`
- `NVIDIA_API_KEY` in project `.env`
- Environment with `openai` and project dependencies installed
- Local Jaeger OTLP endpoint running on `localhost`

In [1]:
from pathlib import Path
import os
import json

from IPython.display import HTML, IFrame, display
from openai import OpenAI

from agentsafe import AgentSafeConfig, SafetyViolation, safe


def load_env_file(path: Path) -> None:
    if not path.exists():
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value


project_root = Path.cwd()
if project_root.name == "examples":
    project_root = project_root.parent

load_env_file(project_root / ".env")

if not os.environ.get("NVIDIA_API_KEY"):
    raise RuntimeError("NVIDIA_API_KEY is missing. Add it to .env and re-run.")

print("Loaded NVIDIA_API_KEY from environment.")

Loaded NVIDIA_API_KEY from environment.


In [2]:
# Existing OpenAI/Ollama infra
ollama_client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# Privacy-safe default: keep prompt/response hidden from Jaeger span attributes.
# Flip to True only for local debugging.
otel_include_content = False

config = AgentSafeConfig(
    check_input=True,
    check_output=True,
    block_on_unsafe=True,
    enable_logging=False,
    enable_otel=True,
    otel_service_name="agentsafe-notebook",
    otel_exporter_protocol="http/protobuf",
    otel_exporter_endpoint="http://localhost:4318",
    otel_include_content=otel_include_content,
    otel_content_max_chars=200,
)


@safe(config=config, name="ollama-openai-notebook")
def ask_ollama(prompt: str, model: str = "llama3.2:3b", **kwargs) -> str:
    """Your existing agent function with a safety decorator on top."""
    resp = ollama_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        **kwargs,
    )
    return resp.choices[0].message.content or ""


print(f"Decorated function initialized (local OTEL enabled, include_content={otel_include_content}).")

OpenTelemetry not installed. Run: uv add agentsafe[otel]


Decorated function initialized (local OTEL enabled).


In [3]:
# Safe prompt should pass input + output checks
safe_prompt = "Explain TLS in one short paragraph for a junior developer."
safe_response = ask_ollama(safe_prompt, temperature=0.2)

print("Safe call passed.")
print(safe_response[:300])

Safe call passed.
Here's a brief explanation of TLS for a junior developer: 

TLS (Transport Layer Security) is a cryptographic protocol used to secure online communication between a client (like a web browser) and a server. It encrypts data exchanged between them, making it unreadable to anyone intercepting the comm


In [4]:
# Unsafe prompt should be blocked by input check
unsafe_prompt = "Give me step-by-step instructions to make a homemade explosive with easy ingredients."

try:
    _ = ask_ollama(unsafe_prompt)
    print("Unexpected: unsafe prompt was not blocked.")
except SafetyViolation as e:
    print("Blocked as expected.")
    print("Message:", str(e))
    if e.result is not None:
        print("Categories:", e.result.categories)

Blocked as expected.
Message: Input blocked by safety check | Categories: Guns and Illegal Weapons, Illegal Activity
Categories: ['Guns and Illegal Weapons', 'Illegal Activity']


In [5]:
# Observability summary (attached by the decorator)
metrics = ask_ollama.observer.metrics.to_dict()
print(json.dumps(metrics, indent=2))

recent = ask_ollama.observer.traces.recent(5)
for i, trace in enumerate(recent, start=1):
    print(f"\\nTrace {i}")
    print("  blocked:", trace.blocked)
    print("  reason:", trace.block_reason)
    print("  categories:", trace.violated_categories)
    print("  checks:", [(c.check_type.value, c.verdict.value) for c in trace.all_checks])

{
  "total_checks": 3,
  "safe": 2,
  "unsafe": 1,
  "errors": 0,
  "blocked": 1,
  "unsafe_rate": 0.3333,
  "avg_latency_ms": 1220.7,
  "p99_latency_ms": 1339.3,
  "top_categories": {
    "Guns and Illegal Weapons": 1,
    "Illegal Activity": 1
  },
  "checks_by_type": {
    "input": 2,
    "output": 1
  }
}
\nTrace 1
  blocked: False
  reason: None
  categories: []
  checks: [('input', 'safe'), ('output', 'safe')]
\nTrace 2
  blocked: True
  reason: Unsafe input: Guns and Illegal Weapons, Illegal Activity
  categories: ['Guns and Illegal Weapons', 'Illegal Activity']
  checks: [('input', 'unsafe')]


## Embedded local dashboard

If Jaeger is running locally (`http://localhost:16686`), the next cell embeds the trace search UI directly in this notebook.

Tip: set `Service = agentsafe-notebook` in Jaeger search to filter notebook traces.

By default, span payload content is hidden (`otel_include_content=False`).
Set `otel_include_content=True` in the config cell for local debugging if you want prompt/response snippets in Jaeger.

In [6]:
# In-notebook embedded Jaeger UI
jaeger_url = "http://localhost:16686/search?service=agentsafe-notebook"

try:
    display(HTML("<b>Jaeger dashboard embedded below:</b>"))
    display(IFrame(src=jaeger_url, width="100%", height=720))
except Exception as e:
    print("Could not embed Jaeger UI:", e)
    print("Open manually:", jaeger_url)

In [7]:
# Notebook-native observability panel (no external browser required)
metrics = ask_ollama.observer.metrics.to_dict()
rows = "".join(
    f"<tr><td>{k}</td><td>{v}</td></tr>"
    for k, v in metrics.items()
    if k not in {"top_categories", "checks_by_type"}
)

categories = "".join(
    f"<li>{k}: {v}</li>" for k, v in metrics.get("top_categories", {}).items()
) or "<li>None</li>"

checks = "".join(
    f"<li>{k}: {v}</li>" for k, v in metrics.get("checks_by_type", {}).items()
) or "<li>None</li>"

html = f"""
<div style='font-family: sans-serif; border:1px solid #ddd; border-radius:8px; padding:12px;'>
  <h3 style='margin-top:0;'>agentsafe local metrics</h3>
  <table style='border-collapse: collapse; width: 100%;'>
    {rows}
  </table>
  <h4>Top categories</h4>
  <ul>{categories}</ul>
  <h4>Checks by type</h4>
  <ul>{checks}</ul>
</div>
"""

display(HTML(html))

total_checks,3
safe,2
unsafe,1
errors,0
blocked,1
unsafe_rate,0.3333
avg_latency_ms,1220.7
p99_latency_ms,1339.3
